# CS224W milestone: rel-trial baselines

**Project:** *When Does Relational Structure Pay Off? Accuracy–Cost Tradeoffs of Relational Graph Models for Clinical Trial Outcome Prediction* (Manel Bermad).

This notebook reproduces every milestone result from the [project repository](https://github.com/mbermad/relational-trials):

1. **Explore** RelBench's `rel-trial` database (schema, label statistics, figures).
2. **LightGBM baselines**: the trial's own table only, and with leakage-safe relational aggregates.
3. **RDL baseline**: heterogeneous GraphSAGE with temporal neighbor sampling.
4. **Results table**: mean ± 95% CI over 3 seeds, next to RelBench's published numbers.

**Before you start:** *Runtime → Change runtime type → GPU* (a T4 is enough). Total time is roughly 1 to 1.5 hours, most of it building the graph's text embeddings once.

## 1. Setup
Results and caches are saved to Google Drive, so a disconnected session can pick up where it left off. Set `USE_DRIVE = False` to keep everything in the session instead.

In [ ]:
USE_DRIVE = True
REPO_URL = "https://github.com/mbermad/relational-trials.git"

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/cs224w-relational-trials"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
RESULTS = f"{WORK}/results"
CACHE = f"{WORK}/cache"
print("Results ->", RESULTS)

In [ ]:
%cd /content
!rm -rf relational-trials && git clone -q {REPO_URL}
%cd /content/relational-trials
!pip install -q -e . torch_geometric pytorch_frame sentence-transformers

PyG's temporal neighbor sampler needs `pyg-lib`, built for the exact torch and CUDA versions Colab ships. The next cell picks the matching wheel.

In [ ]:
import torch
TORCH = torch.__version__.split("+")[0]
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
print("torch", TORCH, "| CUDA", CUDA)
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
import pyg_lib, torch_geometric, torch_frame, relbench
print("pyg-lib", pyg_lib.__version__, "| PyG", torch_geometric.__version__, "| relbench", relbench.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch the runtime to GPU)")

Quick check before the long runs: the test suite (about 1 minute) trains every model on a tiny synthetic dataset with the same schema, including the GNN. If it passes, the real runs below will work.

In [ ]:
!pytest -q

## 2. Explore the data
Loads `rel-trial` from the Hugging Face Hub (first run downloads it), then writes a schema summary, label statistics and four figures to `results/explore/`.

In [ ]:
!python scripts/run.py explore --results {RESULTS}

In [ ]:
from IPython.display import Image, display
for f in ["trials_per_year", "outcome_rate_by_year", "adverse_distribution", "missingness"]:
    display(Image(f"{RESULTS}/explore/{f}.png", width=650))

## 3. LightGBM baselines
Two feature sets, each tuned with the same budget (20 random-search trials on validation), then trained with 3 seeds:

* `table`: the trial's own row in `studies` plus a TF-IDF/SVD summary of its title;
* `agg`: `table` + one-to-one satellite tables (`designs`, `eligibilities`), counts of linked rows, and each linked sponsor/condition/facility/intervention's history. A row with seed time *t* only sees rows dated ≤ *t* and labels known by *t* (label time + 1 year), the same rule the GNN's sampler applies.

In [ ]:
!python scripts/run.py lgbm --features table agg --seeds 0 1 2 --n_trials 20 --results {RESULTS}

## 4. Relational Deep Learning baseline (hetero GraphSAGE)
Builds the relational entity graph once per task (cached in Drive), then trains 3 seeds with RelBench's reference settings: 2 layers, 128 channels, 128/64 sampled neighbors, 10 epochs, lr 0.005.

If the session runs out of RAM while building the graph, rerun with `--text none` (drops text columns) and note it in the report.

In [ ]:
!python scripts/run.py rdl --text glove --seeds 0 1 2 --save_embeddings --cache_dir {CACHE} --results {RESULTS}

## 5. Results table
Mean ± 95% CI over seeds; best value per column in bold; RelBench's published single-run numbers for reference.

In [ ]:
!python scripts/run.py tables --results {RESULTS}
from IPython.display import Markdown
display(Markdown(open(f"{RESULTS}/tables/main.md").read()))

## 6. Package the results
Downloads `milestone_results.zip`. Upload its `explore/` and `tables/` folders to Overleaf (next to `milestone.tex`) and commit them to the repo under `results/`.

In [ ]:
import shutil
shutil.make_archive("/content/milestone_results", "zip", RESULTS, ".")
from google.colab import files
files.download("/content/milestone_results.zip")